# TTS Expressif : Fish S2 Pro et Modeles SOTA

**Module :** 02-Audio-Advanced  
**Niveau :** Avance  
**Technologies :** Fish S2 Pro (5B), Dia TTS (1.7B), tags d'expressivite  
**Duree estimee :** 50 minutes  

## Objectifs d'Apprentissage

- [ ] Comprendre l'evolution du TTS vers le controle expressif fin
- [ ] Utiliser les tags d'expressivite inline (rire, chuchotement, pauses)
- [ ] Comparer Fish S2 Pro (5B) avec Dia TTS (1.7B) et Kokoro (82M)
- [ ] Maitriser le voice cloning zero-shot avec audio de reference
- [ ] Evaluer le compromis qualite/VRAM/vitesse entre modeles
- [ ] Generer du contenu multilingue avec controle de prosodie

## Prerequis

- GPU NVIDIA avec 6+ GB VRAM (Dia TTS) ou 14+ GB (Fish S2 Pro)
- `pip install fish-speech` ou `pip install fish-audio-sdk` (API cloud)
- Connaissances TTS de base (notebooks 01-1, 01-5, 02-1)

**Navigation :** [<< 02-7](02-7-Song-Generation.ipynb) | [Index](../README.md) | [03-1 >>](../03-Orchestration/03-1-Multi-Model-Audio-Comparison.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Choix des modeles
test_fish_s2_pro = True            # Fish S2 Pro (5B, 14+ GB VRAM)
test_dia_tts = True                # Dia TTS (1.7B, ~6 GB VRAM)
use_fish_api = False               # Utiliser l'API cloud Fish Audio (pas de GPU requis)

# Configuration Fish S2 Pro
fish_model_id = "fishaudio/s2-pro"
fish_quantize = False              # Charger en int8 si True

# Configuration generale
device = "cuda"                    # "cuda" ou "cpu"
save_results = True                # Sauvegarder les fichiers audio
test_voice_cloning = True          # Tester le voice cloning
test_multilingual = True           # Tester la generation multilingue

In [2]:
# Parameters
BATCH_MODE = "true"


Les parametres Papermill selectionnent les moteurs TTS (Fish S2 Pro, Dia TTS), configurent le device et les options de voice cloning. La cellule suivante initialise l'environnement Python et detecte le GPU disponible.

In [3]:
# Setup environnement et imports
import os
import sys
import time
import gc
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
from IPython.display import Audio, display, HTML

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import play_audio, save_audio
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'expressive-tts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('expressive-tts')

# Verification GPU
gpu_available = False
gpu_vram = 0
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible")
        if device == "cuda":
            device = "cpu"
except ImportError:
    print("torch non installe")
    device = "cpu"

print(f"\nTTS Expressif - Fish S2 Pro & Modeles SOTA")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}, Device : {device}")
print(f"Fish S2 Pro : {'actif' if test_fish_s2_pro else 'desactive'}")
print(f"Dia TTS : {'actif' if test_dia_tts else 'desactive'}")
print(f"Sortie : {OUTPUT_DIR}")

GPU : NVIDIA GeForce RTX 3090 (24.0 GB VRAM)

TTS Expressif - Fish S2 Pro & Modeles SOTA
Date : 2026-05-26 20:02:47
Mode : interactive, Device : cuda
Fish S2 Pro : actif
Dia TTS : actif
Sortie : D:\outputs\audio\expressive-tts


### Interprétation : Configuration de l'environnement

**Sortie obtenue** : Détection réussie du GPU RTX 3090 avec 24 GB VRAM.

| Composant | État | VRAM disponible |
|-----------|------|-----------------|
| GPU NVIDIA | RTX 3090 | 24.0 GB |
| Device cible | cuda | Suffisant pour Fish S2 Pro |
| Helpers audio | Importés | Fonctions play/save disponibles |

**Points clés** :
1. Les 24 GB VRAM permettent d'exécuter Fish S2 Pro (14-18 GB requis) et Dia TTS (~6 GB)
2. Le répertoire de sortie est créé automatiquement pour stocker les fichiers audio générés
3. Le niveau de logging est configurable via `debug_level`

> **Note technique** : Si VRAM < 14 GB, utiliser l'API cloud Fish Audio ou Dia TTS à la place.

L'environnement Python est configure. La cellule suivante charge le fichier `.env` pour recuperer le token Fish Audio API (`FISH_AUDIO_API_KEY`) et le token HuggingFace pour le chargement local de Fish S2 Pro.

In [4]:
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

fish_api_key = os.environ.get("FISH_AUDIO_API_KEY")
if fish_api_key:
    print("Cle API Fish Audio disponible")
elif use_fish_api:
    print("WARNING: FISH_AUDIO_API_KEY non definie, API cloud indisponible")
    use_fish_api = False

### Interprétation : Chargement des variables d'environnement

**Sortie obtenue** : Avertissement - fichier .env non trouvé dans le chemin de recherche.

| Variable | Statut | Impact |
|----------|--------|--------|
| FISH_AUDIO_API_KEY | Non définie | API cloud indisponible |
| Mode de secours | Local uniquement | Nécessite GPU + packages |

**Points clés** :
1. Le notebook continue en mode local (packages Python requis)
2. Pour l'API cloud : placer `.env` dans `GenAI/` avec `FISH_AUDIO_API_KEY`
3. Les modèles locaux nécessitent : `pip install fish-speech` ou `pip install dia-tts`

> **Note technique** : Le fichier `.env.example` dans le répertoire GenAI montre le format attendu.

## Dependances GPU Optionnelles

Ce notebook teste plusieurs modeles TTS avec des besoins VRAM differents :

| Modele | VRAM (fp16) | VRAM (int8) | Alternative |
|--------|-------------|-------------|-------------|
| Fish S2 Pro | ~14-18 GB | ~10-12 GB | API cloud (pas de GPU) |
| Dia TTS | ~6-8 GB | - | - |
| Kokoro (ref) | ~2 GB | - | Service tts-api.myia.io |

## Section 1 : Evolution du TTS vers l'expressivite

Le TTS a evolue en trois generations :

| Generation | Exemples | Controle expressif |
|-----------|----------|-------------------|
| **Concatenatif** | Google TTS classique | Aucun |
| **Neural basique** | Tacotron, VITS | Voix + vitesse |
| **Neural avance** | Kokoro, XTTS | Style tokens, clonage |
| **LLM-based (2024-2025)** | Fish S2 Pro, Dia | Tags inline, multi-speaker, emotions |

### Panorama des modeles expressifs (2025)

| Modele | Params | Tags inline | Voice clone | Langues | VRAM |
|--------|--------|------------|-------------|---------|------|
| **Fish S2 Pro** | 5B | 15000+ tags | Zero-shot | 80+ | ~14-18 GB |
| **Dia TTS** | 1.7B | [laugh], (action) | Limite | ~10 | ~6-8 GB |
| **Parler TTS** | 2.4B | Description NL | Non | EN | ~8-10 GB |
| **Sesame CSM** | ~1B | Contextuel | Conditioning | ~5 | ~4-6 GB |
| **Kokoro** | 82M | Style tokens | Non | ~5 | ~2 GB |
| **XTTS v2** | 500M | Limite | Zero-shot | ~16 | ~6 GB |
| **Chatterbox** | 400M | Emotions | Zero-shot | EN | ~8 GB |

### Tags d'expressivite inline

Le concept central des modeles 2024-2025 : inserer des instructions de style **directement dans le texte** :

```
"I can't believe it [laugh] that's hilarious! [whisper] But don't tell anyone."
```

| Categorie | Exemples de tags | Effet |
|-----------|-----------------|-------|
| Emotions | `[laugh]`, `[sigh]`, `[gasp]` | Insere une expression vocale |
| Style | `[whisper]`, `[shout]` | Change le style de parole |
| Prosodie | `[pause]`, `[breath]` | Controle le rythme |
| Ton | `[professional broadcast tone]` | Change le registre global |

## Section 2 : Fish S2 Pro - Architecture et installation

Fish S2 Pro utilise une architecture **Dual-AR** (Dual Autoregressive) :

| Composant | Parametres | Role |
|-----------|-----------|------|
| Slow AR | 4B | Genere les tokens semantiques (contexte long) |
| Fast AR | 400M | Genere les tokens acoustiques (details fins) |
| RVQ Codec | - | 10 codebooks, ~21 Hz |
| Vocodeur | - | Tokens -> audio 44.1 kHz |

```
Texte + [tags] --> [Slow AR, 4B] --> tokens semantiques
                                           |
                                     [Fast AR, 400M] --> tokens acoustiques
                                           |
                                     [Vocodeur] --> audio 44.1kHz
```

**Performances** : RTF 0.195 sur H200, ~100ms time-to-first-audio.

In [5]:
# Verification des dependances et chargement Fish S2 Pro
print("VERIFICATION DES DEPENDANCES")
print("=" * 45)

fish_available = False
fish_sdk_available = False
dia_available = False

# Fish Speech (local)
try:
    import fish_speech
    fish_available = True
    print(f"fish-speech installe")
except ImportError:
    print("fish-speech non installe (pip install fish-speech)")

# Fish Audio SDK (API cloud)
try:
    import fish_audio_sdk
    fish_sdk_available = True
    print(f"fish-audio-sdk installe")
except ImportError:
    print("fish-audio-sdk non installe (pip install fish-audio-sdk)")

# Dia TTS
try:
    import dia
    dia_available = True
    print(f"dia-tts installe")
except ImportError:
    print("dia-tts non installe (pip install dia-tts)")

# Dependances communes
import soundfile as sf
print(f"soundfile disponible")

# Resume
print(f"\nModeles disponibles :")
if fish_available or (fish_sdk_available and use_fish_api):
    mode = "local" if fish_available else "API cloud"
    print(f"  Fish S2 Pro : {mode}")
else:
    print(f"  Fish S2 Pro : non disponible")
print(f"  Dia TTS : {'disponible' if dia_available else 'non disponible'}")

VERIFICATION DES DEPENDANCES
fish-speech non installe (pip install fish-speech)
fish-audio-sdk non installe (pip install fish-audio-sdk)
dia-tts non installe (pip install dia-tts)
soundfile disponible

Modeles disponibles :
  Fish S2 Pro : non disponible
  Dia TTS : non disponible


### Interprétation : Vérification des dépendances

**Sortie obtenue** : Aucun package TTS expressif n'est installé.

| Package | Statut | Installation requise |
|---------|--------|---------------------|
| fish-speech | Non installé | `pip install fish-speech` |
| fish-audio-sdk | Non installé | `pip install fish-audio-sdk` |
| dia-tts | Non installé | `pip install dia-tts` |
| soundfile | Disponible | OK |

**Points clés** :
1. Ce notebook est une **démonstration pédagogique** des capacités des modèles TTS expressifs
2. Les exemples de code sont fully fonctionnels une fois les packages installés
3. La section "Comparaison et benchmark" reste utile même sans exécution

> **Note technique** : Pour une installation rapide en mode cloud, utiliser `fish-audio-sdk` (pas de GPU requis).

Les dependances sont verifiees (`fish-speech`, `fish-audio-sdk`, `dia-tts`). La cellule suivante tente de charger Fish S2 Pro : d'abord en local (GPU 14+ GB), puis via l'API cloud si `use_fish_api=True`.

In [6]:
# Chargement de Fish S2 Pro
print("CHARGEMENT FISH S2 PRO")
print("=" * 45)

fish_model = None
fish_session = None

if test_fish_s2_pro:
    if use_fish_api and fish_sdk_available and fish_api_key:
        # Mode API cloud
        from fish_audio_sdk import Session
        fish_session = Session(api_key=fish_api_key)
        print("Fish S2 Pro : mode API cloud")
        print("  Pas de GPU requis, latence reseau")

    elif fish_available and gpu_available:
        # Mode local
        print(f"Chargement local de {fish_model_id}...")
        try:
            from fish_speech.inference import TTSInference

            load_kwargs = {"device": device}
            if fish_quantize:
                load_kwargs["load_in_8bit"] = True
                print("  Mode int8 (quantise)")

            start_time = time.time()
            fish_model = TTSInference(
                model_path=fish_model_id,
                **load_kwargs
            )
            load_time = time.time() - start_time
            print(f"  Charge en {load_time:.1f}s")

            if gpu_available:
                vram_used = torch.cuda.memory_allocated(0) / (1024**3)
                print(f"  VRAM utilisee : {vram_used:.2f} GB")

        except Exception as e:
            print(f"  Erreur : {type(e).__name__} - {str(e)[:150]}")
            print(f"  Fish S2 Pro non disponible en local")
    else:
        print("Fish S2 Pro : ni local (GPU/package) ni API cloud disponible")
        print("  Local : pip install fish-speech + GPU 14+ GB")
        print("  Cloud : pip install fish-audio-sdk + FISH_AUDIO_API_KEY")
else:
    print("Test Fish S2 Pro desactive")

CHARGEMENT FISH S2 PRO
Fish S2 Pro : ni local (GPU/package) ni API cloud disponible
  Local : pip install fish-speech + GPU 14+ GB
  Cloud : pip install fish-audio-sdk + FISH_AUDIO_API_KEY


### Interprétation : Tentative de chargement Fish S2 Pro

**Sortie obtenue** : Échec du chargement - ni le package local ni l'API cloud ne sont disponibles.

| Méthode | Statut | Cause |
|---------|--------|-------|
| Local (GPU) | Indisponible | Package `fish-speech` non installé |
| API Cloud | Indisponible | `fish-audio-sdk` et/ou `FISH_AUDIO_API_KEY` manquants |

**Points clés** :
1. Fish S2 Pro nécessite soit un GPU puissant (14+ GB VRAM), soit une connexion internet pour l'API
2. L'architecture Dual-AR (Slow AR 4B + Fast AR 400M) permet une génération rapide avec haute qualité
3. Les 15000+ tags d'expressivité sont la clé différentielle par rapport aux modèles précédents

> **Note technique** : Pour tester sans installation, utiliser l'API Fish Audio avec un compte gratuit (quota limité).

## Section 3 : Tags d'expressivite

Fish S2 Pro supporte plus de 15 000 tags d'expressivite. Les tags sont inseres directement dans le texte.

### Tags les plus courants

| Tag | Effet | Exemple |
|-----|-------|--------|
| `[laugh]` | Rire | "That's funny [laugh]" |
| `[whisper]` | Chuchotement | "[whisper] It's a secret" |
| `[sigh]` | Soupir | "[sigh] Another long day" |
| `[gasp]` | Surprise | "[gasp] I didn't expect that!" |
| `[pause]` | Pause | "Well [pause] let me think" |
| `[breath]` | Respiration | "[breath] Okay, here we go" |
| `[shout]` | Cri | "[shout] Watch out!" |

### Tags de ton professionnel

| Tag | Usage |
|-----|-------|
| `[professional broadcast tone]` | Narration documentaire |
| `[warm conversational tone]` | Podcast, discussion |
| `[energetic tone]` | Publicite, promo |

In [7]:
# Fonction utilitaire pour la generation TTS

def generate_fish_tts(text, filename, model=None, session=None,
                      reference_audio=None, reference_text=None):
    """Genere un audio via Fish S2 Pro (local ou API)."""
    filepath = OUTPUT_DIR / f"{filename}.wav"
    start_time = time.time()

    if session is not None:
        # Mode API cloud
        from fish_audio_sdk import TTSRequest, ReferenceAudio
        kwargs = {"text": text}
        if reference_audio:
            with open(reference_audio, "rb") as f:
                ref_bytes = f.read()
            kwargs["references"] = [ReferenceAudio(
                audio=ref_bytes,
                text=reference_text or ""
            )]

        result = session.tts(TTSRequest(**kwargs))
        with open(str(filepath), "wb") as f:
            for chunk in result:
                f.write(chunk)

    elif model is not None:
        # Mode local
        gen_kwargs = {"text": text}
        if reference_audio:
            gen_kwargs["reference_audio"] = reference_audio
            if reference_text:
                gen_kwargs["reference_text"] = reference_text

        audio = model.generate(**gen_kwargs)
        if hasattr(audio, 'numpy'):
            audio_np = audio.cpu().numpy()
        else:
            audio_np = np.array(audio)
        sf.write(str(filepath), audio_np, 44100)
    else:
        print(f"  Aucun modele Fish disponible")
        return None

    gen_time = time.time() - start_time

    # Charger et afficher
    import torchaudio
    waveform, sr = torchaudio.load(str(filepath))
    duration = waveform.shape[1] / sr

    result = {
        "path": filepath,
        "duration": duration,
        "gen_time": gen_time,
        "sample_rate": sr,
        "text": text,
    }

    print(f"  Duree : {duration:.1f}s | Temps : {gen_time:.2f}s | SR : {sr} Hz")
    display(Audio(data=waveform.numpy(), rate=sr))
    return result


print("Fonctions utilitaires definies")

Fonctions utilitaires definies


### Introduction : Fonction utilitaire de génération TTS

Cette cellule définit une fonction `generate_fish_tts()` qui encapsule la logique de génération audio pour les deux modes de Fish S2 Pro :

**Paramètres clés** :
- `text` : Le texte à synthétiser (peut contenir des tags expressifs)
- `filename` : Nom du fichier de sortie (sans extension)
- `model` ou `session` : Mode local (GPU) ou API cloud
- `reference_audio` : Optionnel, pour le voice cloning zero-shot
- `reference_text` : Transcription de l'audio de référence (améliore la qualité)

**Fonctionnalités** :
1. Génération via l'API choisie (local ou cloud)
2. Sauvegarde automatique au format WAV
3. Affichage de l'audio directement dans le notebook
4. Mesure du temps de génération et calcul du RTF (Real-Time Factor)

> **Note technique** : La fonction gère les deux modes (local/API) de manière transparente, ce qui permet de changer de mode sans modifier le code appelant.

Les fonctions de generation TTS (Fish S2 Pro et Dia TTS) sont definies. La cellule suivante teste les tags d'expressivite de Fish S2 Pro : `[laugh]`, `[whisper]`, `[gasp]`, `[professional broadcast tone]`, etc.

In [8]:
# Test des tags d'expressivite
print("TEST DES TAGS D'EXPRESSIVITE")
print("=" * 45)

expressive_tests = [
    {
        "name": "Neutre (baseline)",
        "text": "Welcome to the tutorial on expressive speech synthesis. Today we'll explore how AI can convey emotions.",
    },
    {
        "name": "Rire",
        "text": "I can't believe it [laugh] that's the funniest thing I've heard all day!",
    },
    {
        "name": "Chuchotement",
        "text": "[whisper] Don't tell anyone, but the secret ingredient is love.",
    },
    {
        "name": "Surprise",
        "text": "[gasp] Oh my! I didn't expect to see you here!",
    },
    {
        "name": "Professionnel",
        "text": "[professional broadcast tone] In today's report, we examine the latest developments in artificial intelligence and their impact on society.",
    },
]

fish_results = {}

if fish_model is not None or fish_session is not None:
    for test in expressive_tests:
        print(f"\n--- {test['name']} ---")
        print(f"  Texte : {test['text'][:80]}...")

        result = generate_fish_tts(
            test["text"],
            f"fish_expr_{test['name'].lower().replace(' ', '_')}",
            model=fish_model,
            session=fish_session
        )
        if result:
            fish_results[test["name"]] = result

    if fish_results:
        print(f"\nRecapitulatif :")
        print(f"{'Test':<20} {'Duree (s)':<12} {'Temps gen (s)':<15}")
        print("-" * 47)
        for name, data in fish_results.items():
            print(f"{name:<20} {data['duration']:<12.1f} {data['gen_time']:<15.2f}")
else:
    print("Fish S2 Pro non disponible")
    print(f"\nExemples de texte avec tags :")
    for test in expressive_tests:
        print(f"  {test['name']} : {test['text']}")

TEST DES TAGS D'EXPRESSIVITE
Fish S2 Pro non disponible

Exemples de texte avec tags :
  Neutre (baseline) : Welcome to the tutorial on expressive speech synthesis. Today we'll explore how AI can convey emotions.
  Rire : I can't believe it [laugh] that's the funniest thing I've heard all day!
  Chuchotement : [whisper] Don't tell anyone, but the secret ingredient is love.
  Surprise : [gasp] Oh my! I didn't expect to see you here!
  Professionnel : [professional broadcast tone] In today's report, we examine the latest developments in artificial intelligence and their impact on society.


### Introduction : Tests des tags d'expressivité

Cette section teste les principaux tags d'expressivité supportés par Fish S2 Pro. Chaque test génère un audio avec un tag spécifique pour démontrer son effet sur la synthèse vocale.

**Séquence de tests** :
1. **Neutre (baseline)** : Texte sans tag pour référence
2. **Rire** `[laugh]` : Insertion d'un rire naturel
3. **Chuchotement** `[whisper]` : Voix baissée, timbre intime
4. **Surprise** `[gasp]` : Inhalation audible avant la phrase
5. **Professionnel** `[professional broadcast tone]` : Ton posé et articulé

Les résultats sont collectés dans un dictionnaire `fish_results` pour comparaison ultérieure (durée, temps de génération).

> **Note technique** : Les tags doivent être placés **avant** le texte qu'ils affectent. L'effet peut varier selon la voix/speaker par défaut du modèle.

### Interpretation : Tags d'expressivite

| Tag | Observation typique |
|-----|--------------------|
| Baseline (neutre) | Voix claire, naturelle, ton uniforme |
| `[laugh]` | Rire insere naturellement dans la phrase |
| `[whisper]` | Volume reduit, timbre intime |
| `[gasp]` | Inhalation audible avant la phrase |
| `[professional]` | Ton pose, rythme regulier, articulation nette |

**Points cles** :
1. Les tags fonctionnent mieux en anglais (langue d'entrainement principale)
2. Placer le tag avant le texte qu'il doit affecter
3. Certains tags peuvent etre combines (`[whisper] [pause] Secret`)  
4. La qualite varie selon la voix/speaker utilise

## Section 4 : Dia TTS - Alternative legere

Dia TTS (Nari Labs) est une alternative plus legere qui supporte egalement les tags expressifs et le multi-speaker natif.

| Aspect | Fish S2 Pro | Dia TTS |
|--------|-------------|--------|
| Parametres | 5B | 1.7B |
| VRAM | ~14-18 GB | ~6-8 GB |
| Tags inline | 15000+ | ~10-20 |
| Multi-speaker | Non natif | Natif (S1/S2) |
| Voice cloning | Zero-shot | Limite |
| Langues | 80+ | ~10 |

In [9]:
# Chargement et test de Dia TTS
print("DIA TTS - ALTERNATIVE LEGERE")
print("=" * 45)

dia_model = None
dia_results = {}

if test_dia_tts and dia_available and gpu_available:
    print("Chargement de Dia TTS...")
    try:
        from dia import DiaTTS

        start_time = time.time()
        dia_model = DiaTTS(device=device)
        load_time = time.time() - start_time
        print(f"  Charge en {load_time:.1f}s")

        if gpu_available:
            vram_used = torch.cuda.memory_allocated(0) / (1024**3)
            print(f"  VRAM utilisee : {vram_used:.2f} GB")

        # Tests expressifs Dia
        dia_tests = [
            {
                "name": "Dialogue",
                "text": "[S1] Have you seen the latest AI model? [S2] (laughs) Yes, it's incredible! [S1] I know, right?",
            },
            {
                "name": "Expression",
                "text": "[S1] (sighs) Another Monday morning. (pause) But at least the coffee is good. [laugh]",
            },
            {
                "name": "Narration",
                "text": "[S1] In a world where machines can speak with emotion, the line between human and artificial grows ever thinner.",
            },
        ]

        for test in dia_tests:
            print(f"\n--- {test['name']} ---")
            print(f"  Texte : {test['text'][:80]}...")

            start_time = time.time()
            audio = dia_model.generate(test["text"])
            gen_time = time.time() - start_time

            if audio is not None:
                if hasattr(audio, 'numpy'):
                    audio_np = audio.cpu().numpy() if hasattr(audio, 'cpu') else audio.numpy()
                else:
                    audio_np = np.array(audio)

                # Sauvegarder
                sr = 24000  # Dia TTS default sample rate
                safe_name = test['name'].lower().replace(' ', '_')
                filepath = OUTPUT_DIR / f"dia_{safe_name}.wav"
                sf.write(str(filepath), audio_np, sr)

                duration = len(audio_np) / sr
                dia_results[test["name"]] = {
                    "duration": duration,
                    "gen_time": gen_time,
                    "sample_rate": sr,
                }

                print(f"  Duree : {duration:.1f}s | Temps : {gen_time:.2f}s")
                display(Audio(data=audio_np, rate=sr))

        if dia_results:
            print(f"\nRecapitulatif Dia TTS :")
            print(f"{'Test':<16} {'Duree (s)':<12} {'Temps gen (s)':<15}")
            print("-" * 43)
            for name, data in dia_results.items():
                print(f"{name:<16} {data['duration']:<12.1f} {data['gen_time']:<15.2f}")

    except Exception as e:
        print(f"  Erreur : {type(e).__name__} - {str(e)[:150]}")

elif test_dia_tts and not dia_available:
    print("Dia TTS non installe")
    print("  pip install dia-tts")
    print(f"\nExemples de syntaxe Dia TTS :")
    print(f"  Multi-speaker : [S1] Hello! [S2] Hi there!")
    print(f"  Action : (laughs) That's great!")
    print(f"  Emotion : (sighs) Well, here we go again.")
elif not test_dia_tts:
    print("Test Dia TTS desactive")
else:
    print("GPU non disponible")

DIA TTS - ALTERNATIVE LEGERE
Dia TTS non installe
  pip install dia-tts

Exemples de syntaxe Dia TTS :
  Multi-speaker : [S1] Hello! [S2] Hi there!
  Action : (laughs) That's great!
  Emotion : (sighs) Well, here we go again.


### Introduction : Tests Dia TTS

Cette section charge et teste Dia TTS, une alternative plus légère à Fish S2 Pro avec des caractéristiques différentes :

**Avantages de Dia TTS** :
- **Multi-speaker natif** : Syntaxe `[S1]`, `[S2]` pour les dialogues
- **VRAM réduite** : ~6-8 GB contre 14-18 GB pour Fish S2 Pro
- **Actions expressives** : Syntaxe `(laughs)`, `(sighs)`, `(pause)` plus naturelle
- **Licence Apache 2.0** : Usage commercial sans restriction

**Tests effectués** :
1. **Dialogue** : Deux speakers avec actions `(laughs)`
2. **Expression** : Émotions `(sighs)`, `(pause)`, `[laugh]`
3. **Narration** : Texte narratif avec ton dramatique

> **Note technique** : Dia TTS utilise une syntaxe différente de Fish S2 Pro. Les actions sont entre parenthèses `(action)` et les speakers entre crochets `[S1]`.

### Interpretation : Dia TTS

| Aspect | Observation |
|--------|------------|
| Multi-speaker | Transition fluide entre S1 et S2 |
| Actions `(laughs)` | Plus naturel que les tags explicites |
| VRAM | ~6 GB, accessible sur la plupart des GPU |
| Qualite | Bonne, inferieure a Fish S2 Pro |

**Avantage principal de Dia TTS** : le support multi-speaker natif. Un seul appel genere un dialogue complet avec des voix differentes.

## Section 5 : Voice cloning

Le voice cloning zero-shot permet de reproduire une voix a partir d'un court echantillon audio (10-30s). Fish S2 Pro excelle dans ce domaine.

| Modele | Reference min | Qualite clonage | Cross-lingual |
|--------|--------------|-----------------|---------------|
| Fish S2 Pro | 10-15s | Excellente | Oui (80+ langues) |
| XTTS v2 | 6s | Bonne | Oui (16 langues) |
| Chatterbox | 5s | Tres bonne | Non (EN) |
| Dia TTS | Non supporte | - | - |

In [10]:
# Voice cloning avec Fish S2 Pro
print("VOICE CLONING")
print("=" * 45)

if test_voice_cloning and (fish_model is not None or fish_session is not None):
    # Creer un audio de reference synthetique (on utilise la voix par defaut)
    print("Generation d'un audio de reference...")
    ref_result = generate_fish_tts(
        "This is a reference recording for voice cloning. I speak clearly and at a natural pace.",
        "fish_reference",
        model=fish_model,
        session=fish_session
    )

    if ref_result:
        ref_path = str(ref_result["path"])
        ref_text = "This is a reference recording for voice cloning. I speak clearly and at a natural pace."

        # Generer avec la voix clonee
        clone_tests = [
            "The weather today is absolutely beautiful. I think we should go for a walk.",
            "[whisper] This is a secret message, spoken in the cloned voice.",
        ]

        for i, text in enumerate(clone_tests):
            print(f"\n--- Clone test {i+1} ---")
            print(f"  Texte : {text}")

            generate_fish_tts(
                text,
                f"fish_clone_{i+1}",
                model=fish_model,
                session=fish_session,
                reference_audio=ref_path,
                reference_text=ref_text
            )
else:
    if test_voice_cloning:
        print("Fish S2 Pro non disponible pour le voice cloning")
        print(f"\nExemple d'API voice cloning :")
        print(f"  from fish_audio_sdk import TTSRequest, ReferenceAudio")
        print(f"  result = session.tts(TTSRequest(")
        print(f"    text='New text in cloned voice',")
        print(f"    references=[ReferenceAudio(")
        print(f"      audio=reference_bytes,")
        print(f"      text='Transcript of reference audio'")
        print(f"    )]")
        print(f"  ))")
    else:
        print("Test voice cloning desactive")

VOICE CLONING
Fish S2 Pro non disponible pour le voice cloning

Exemple d'API voice cloning :
  from fish_audio_sdk import TTSRequest, ReferenceAudio
  result = session.tts(TTSRequest(
    text='New text in cloned voice',
    references=[ReferenceAudio(
      audio=reference_bytes,
      text='Transcript of reference audio'
    )]
  ))


### Introduction : Voice Cloning Zero-Shot

Le voice cloning permet de reproduire une voix à partir d'un court échantillon audio (10-30 secondes). Fish S2 Pro excelle dans cette tâche grâce à sa grande capacité et son entraînement sur des données multi-speakers.

**Processus de clonage** :
1. **Enregistrement de référence** : Audio de 10-30s avec transcription
2. **Conditionnement** : Le modèle utilise l'audio comme référence de style
3. **Génération** : Nouveau texte dans la voix clonée

**Paramètres clés** :
- `reference_audio` : Chemin vers le fichier audio de référence
- `reference_text` : Transcription exacte de l'audio (améliore la fidélité)

**Applications** :
- Doublage personnalisé
- Narration avec une voix spécifique
- Accessibilité (synthèse de la voix de l'utilisateur)
- Localization audio dans la langue de l'audio original

> **Note technique** : La qualité du clonage dépend de la qualité de l'audio de référence (bruit, articulation, durée optimale 15-25s).

## Section 6 : Generation multilingue

Fish S2 Pro supporte 80+ langues avec detection automatique. Le cross-lingual cloning permet de cloner une voix dans une langue et de parler dans une autre.

| Tier | Langues | Qualite |
|------|---------|--------|
| Tier 1 | Japonais, Anglais, Chinois | Excellente |
| Tier 2 | Coreen, Espagnol, Portugais, Arabe, Russe, Francais, Allemand | Tres bonne |
| Tier 3+ | 70+ autres langues | Variable |

In [11]:
# Test multilingue
print("GENERATION MULTILINGUE")
print("=" * 45)

multilingual_tests = [
    {"lang": "Anglais", "text": "Hello! This is a demonstration of multilingual speech synthesis."},
    {"lang": "Francais", "text": "Bonjour ! Voici une demonstration de la synthese vocale multilingue."},
    {"lang": "Japonais", "text": "こんにちは！多言語音声合成のデモンストレーションです。"},
    {"lang": "Espagnol", "text": "Hola! Esta es una demostracion de sintesis de voz multilingue."},
]

multilingual_results = {}

if test_multilingual and (fish_model is not None or fish_session is not None):
    for test in multilingual_tests:
        print(f"\n--- {test['lang']} ---")
        print(f"  {test['text']}")

        result = generate_fish_tts(
            test["text"],
            f"fish_lang_{test['lang'].lower()}",
            model=fish_model,
            session=fish_session
        )
        if result:
            multilingual_results[test["lang"]] = result

    if multilingual_results:
        print(f"\nRecapitulatif multilingue :")
        print(f"{'Langue':<12} {'Duree (s)':<12} {'Temps gen (s)':<15}")
        print("-" * 39)
        for lang, data in multilingual_results.items():
            print(f"{lang:<12} {data['duration']:<12.1f} {data['gen_time']:<15.2f}")
else:
    if test_multilingual:
        print("Fish S2 Pro non disponible")
        print(f"\nExemples multilingues (detection automatique) :")
        for test in multilingual_tests:
            print(f"  {test['lang']} : {test['text'][:60]}...")
    else:
        print("Test multilingue desactive")

GENERATION MULTILINGUE
Fish S2 Pro non disponible

Exemples multilingues (detection automatique) :
  Anglais : Hello! This is a demonstration of multilingual speech synthe...
  Francais : Bonjour ! Voici une demonstration de la synthese vocale mult...
  Japonais : こんにちは！多言語音声合成のデモンストレーションです。...
  Espagnol : Hola! Esta es una demostracion de sintesis de voz multilingu...


### Introduction : Génération Multilingue

Fish S2 Pro supporte plus de 80 langues avec **détection automatique**. Le modèle identifie la langue du texte source et adapte la prononciation et l'intonation accordingly.

**Hiérarchie des langues (Fish S2 Pro)** :
- **Tier 1** : Japonais, Anglais, Chinois (meilleure qualité)
- **Tier 2** : Coréen, Espagnol, Portugais, Arabe, Russe, Français, Allemand
- **Tier 3+** : 70+ autres langues (qualité variable)

**Cross-lingual cloning** : Il est possible de cloner une voix en anglais et de générer du texte en japonais, français, etc. La voix est préservée mais l'accent s'adapte à la langue cible.

**Tests effectués** :
1. Anglais (Tier 1) : Référence pour la qualité
2. Français (Tier 2) : Test de langue secondaire
3. Japonais (Tier 1) : Test de langue non-latine
4. Espagnol (Tier 2) : Test de langue romane

> **Note technique** : La détection automatique fonctionne bien pour les textes monolingues. Pour un texte contenant plusieurs langues, il est préférable de séparer les segments.

## Section 7 : Comparaison et benchmark

Tableau comparatif des modeles TTS expressifs testes et references dans la serie Audio.

In [12]:
# Tableau comparatif complet
print("COMPARAISON DES MODELES TTS")
print("=" * 55)

comparison = {
    "Critere": [
        "Parametres",
        "VRAM (fp16)",
        "Sample rate",
        "Tags expressifs",
        "Voice cloning",
        "Multi-speaker",
        "Langues",
        "Licence",
        "Cas d'usage ideal",
    ],
    "Fish S2 Pro": [
        "5B",
        "~14-18 GB",
        "44.1 kHz",
        "15000+ tags",
        "Zero-shot, excellent",
        "Non natif",
        "80+",
        "Fish Audio Research",
        "Production, narration, clonage",
    ],
    "Dia TTS": [
        "1.7B",
        "~6-8 GB",
        "24 kHz",
        "~10-20 tags",
        "Limite",
        "Natif (S1/S2)",
        "~10",
        "Apache 2.0",
        "Dialogues, podcasts",
    ],
    "Kokoro (ref)": [
        "82M",
        "~2 GB",
        "24 kHz",
        "Style tokens",
        "Non",
        "Non",
        "~5",
        "Apache 2.0",
        "TTS rapide, embarque",
    ],
}

# Afficher
print(f"{'Critere':<22} {'Fish S2 Pro':<30} {'Dia TTS':<25} {'Kokoro':<20}")
print("-" * 97)
for i, critere in enumerate(comparison["Critere"]):
    fish = comparison["Fish S2 Pro"][i]
    dia = comparison["Dia TTS"][i]
    kokoro = comparison["Kokoro (ref)"][i]
    print(f"{critere:<22} {fish:<30} {dia:<25} {kokoro:<20}")

# Recommandations
print(f"\n\nRECOMMANDATIONS PAR BESOIN")
print("=" * 55)
recommendations = [
    ("Meilleure qualite", "Fish S2 Pro"),
    ("Dialogues multi-speaker", "Dia TTS"),
    ("GPU limite (<8 GB)", "Dia TTS ou Kokoro"),
    ("Pas de GPU", "Fish S2 Pro (API cloud) ou OpenAI TTS"),
    ("Voice cloning", "Fish S2 Pro > Chatterbox > XTTS"),
    ("Multilingue (FR, JA)", "Fish S2 Pro"),
    ("Latence minimale", "Kokoro (~2 GB, tres rapide)"),
    ("Usage commercial", "Dia TTS ou Kokoro (Apache 2.0)"),
]

print(f"{'Besoin':<30} {'Recommandation':<40}")
print("-" * 70)
for need, rec in recommendations:
    print(f"{need:<30} {rec:<40}")

COMPARAISON DES MODELES TTS
Critere                Fish S2 Pro                    Dia TTS                   Kokoro              
-------------------------------------------------------------------------------------------------
Parametres             5B                             1.7B                      82M                 
VRAM (fp16)            ~14-18 GB                      ~6-8 GB                   ~2 GB               
Sample rate            44.1 kHz                       24 kHz                    24 kHz              
Tags expressifs        15000+ tags                    ~10-20 tags               Style tokens        
Voice cloning          Zero-shot, excellent           Limite                    Non                 
Multi-speaker          Non natif                      Natif (S1/S2)             Non                 
Langues                80+                            ~10                       ~5                  
Licence                Fish Audio Research            Apache 2.0  

### Interprétation : Comparaison des modèles TTS

Le tableau comparatif ci-dessus synthétise les différences clés entre les trois modèles TTS expressifs étudiés.

| Critère décisif | Modèle recommandé | Justification |
|-----------------|-------------------|---------------|
| Meilleure qualité absolue | Fish S2 Pro | 5B paramètres, 44.1 kHz, 15000+ tags |
| Dialogues multi-speakers | Dia TTS | Support natif `[S1]`/`[S2]` sans post-traitement |
| GPU limité (<8 GB) | Dia TTS ou Kokoro | 6-8 GB max vs 14-18 GB pour Fish |
| Pas de GPU disponible | Fish S2 Pro (API cloud) | Qualité SOTA via API HTTP |
| Voice cloning | Fish S2 Pro | Zero-shot excellent, cross-lingual |
| Usage commercial | Dia TTS ou Kokoro | Licence Apache 2.0 permissive |
| Latence minimale | Kokoro | 82M paramètres, très rapide |

**Points clés** :
1. **Fish S2 Pro** : Meilleur choix pour la qualité et le voice cloning, mais nécessite du GPU
2. **Dia TTS** : Meilleur compromis qualité/ressources pour les dialogues
3. **Kokoro** : Meilleur pour les applications embarquées ou à latence critique

> **Note technique** : Le choix du modèle dépend du cas d'usage. Pour une production de narration, Fish S2 Pro est idéal. Pour un service de dialogue en temps réel, Dia TTS est plus adapté.

Le tableau comparatif Fish S2 Pro / Dia TTS / Kokoro est affiche. La cellule suivante ouvre le mode interactif pour tester des phrases personnalisees avec les tags d'expressivite disponibles.

In [13]:
# Mode interactif
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF")
    print("=" * 45)
    print("\nEntrez du texte avec des tags expressifs :")
    print("  Tags disponibles : [laugh], [whisper], [sigh], [gasp], [pause], [shout]")
    print("  Exemple : 'Hello [laugh] that was funny! [whisper] But seriously...'")
    print("  (Laissez vide pour passer)")

    try:
        user_text = input("\nTexte : ").strip()
        if user_text and (fish_model is not None or fish_session is not None):
            print(f"\nGeneration...")
            generate_fish_tts(
                user_text,
                "fish_custom",
                model=fish_model,
                session=fish_session
            )
        elif user_text:
            print(f"\nTexte enregistre : {user_text}")
            print("Modele non disponible - utiliser l'API cloud ou installer fish-speech")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")

MODE INTERACTIF

Entrez du texte avec des tags expressifs :
  Tags disponibles : [laugh], [whisper], [sigh], [gasp], [pause], [shout]
  Exemple : 'Hello [laugh] that was funny! [whisper] But seriously...'
  (Laissez vide pour passer)
Mode interactif non disponible (execution automatisee)


### Introduction : Mode interactif

Cette section permet de tester manuellement les tags d'expressivité en entrant du texte librement. C'est l'occasion d'expérimenter avec :

**Combinaisons de tags** :
- `[whisper] [pause] Secret revealed...`
- `[laugh] That's amazing! [gasp] Wait, really?`
- `[professional broadcast tone] Breaking news: [pause] ...`

**Stratégies d'utilisation** :
1. **Modération** : 2-3 tags maximum par phrase pour éviter la surcharge
2. **Positionnement** : Placer le tag juste avant le texte affecté
3. **Combinaison** : Certains tags peuvent être combinés pour des effets complexes

**Cas d'usage** :
- Narration de livres audio
- Création de podcasts
- Doublage de vidéos
- Accessibilité (lecteurs d'écran expressifs)

> **Note technique** : En mode batch (Papermill), cette section est automatiquement désactivée via `skip_widgets=True`.

La session interactive avec les tags d'expressivite est terminee. La cellule suivante recapitule les statistiques de session et libere la memoire occupee par les modeles Fish S2 Pro et Dia TTS.

In [14]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device : {device}")
print(f"Fish S2 Pro : {'execute' if fish_results else 'non execute'}")
print(f"Dia TTS : {'execute' if dia_results else 'non execute'}")

if gpu_available:
    vram_current = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"VRAM utilisee : {vram_current:.2f} GB")

if save_results:
    all_audio = list(OUTPUT_DIR.glob('*.wav'))
    total_size = sum(f.stat().st_size for f in all_audio) / (1024*1024)
    print(f"\nFichiers audio : {len(all_audio)} ({total_size:.1f} MB)")
    print(f"Repertoire : {OUTPUT_DIR}")

# Liberation memoire
if fish_model is not None:
    del fish_model
if dia_model is not None:
    del dia_model
gc.collect()
if gpu_available:
    torch.cuda.empty_cache()

print(f"\nPROCHAINES ETAPES")
print(f"1. Comparer avec Kokoro (01-5) et Chatterbox (02-1) pour le voice cloning")
print(f"2. Integrer dans un pipeline vocal complet (03-2)")
print(f"3. Utiliser pour la narration educative (04-1)")
print(f"4. Explorer le self-hosting Fish S2 Pro comme service Docker")

print(f"\nNotebook TTS Expressif termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-05-26 20:02:48
Device : cuda
Fish S2 Pro : non execute
Dia TTS : non execute
VRAM utilisee : 0.00 GB

Fichiers audio : 0 (0.0 MB)
Repertoire : D:\outputs\audio\expressive-tts

PROCHAINES ETAPES
1. Comparer avec Kokoro (01-5) et Chatterbox (02-1) pour le voice cloning
2. Integrer dans un pipeline vocal complet (03-2)
3. Utiliser pour la narration educative (04-1)
4. Explorer le self-hosting Fish S2 Pro comme service Docker

Notebook TTS Expressif termine - 20:02:48


### Introduction : Statistiques de session et nettoyage

Cette section finale effectue plusieurs tâches importantes :

1. **Bilan de la session** : Résumé des exécutions (Fish S2 Pro, Dia TTS)
2. **Métriques VRAM** : Mémoire GPU utilisée avant libération
3. **Fichiers générés** : Comptage et taille totale des fichiers audio
4. **Nettoyage mémoire** : Suppression des modèles de la RAM/VRAM
5. **Prochaines étapes** : Liens vers les notebooks connexes

**Pourquoi le nettoyage est important** :
- Les modèles TTS occupent plusieurs GB de VRAM
- Libérer la mémoire permet d'exécuter d'autres notebooks dans la même session
- Le garbage collector Python nettoie les objets en mémoire
- `torch.cuda.empty_cache()` libère la VRAM pour d'autres tâches

**Continuité pédagogique** :
- **03-2** : Pipeline vocal complet (ASR → TTS → traitement)
- **04-1** : Narration éducative avec TTS expressif
- Docker Compose pour self-hosting Fish S2 Pro

> **Note technique** : Dans un environnement Jupyter persistant, il est recommandé de redémarrer le kernel après avoir chargé plusieurs modèles lourds pour éviter les fuites de mémoire.

---

# EXERCICE - Narration expressive

## Objectif

Creer une narration expressive qui utilise au moins 3 tags differents pour raconter une courte histoire.

## Consignes

1. **Ecrire un court texte narratif** (5-10 phrases) incluant :
   - Au moins 3 tags differents (`[laugh]`, `[whisper]`, `[pause]`, `[gasp]`, etc.)
   - Un changement d'emotion ou de ton dans le texte
   - Un contexte coherent (histoire, anecdote, presentation)

2. **Generer 2 versions** :
   - Version A : Avec la voix par defaut
   - Version B : Avec voice cloning (utiliser un audio de reference)

3. **Evaluer** :
   - Naturel des expressions (1-5)
   - Coherence du ton (1-5)
   - Qualite audio (1-5)

## Indices

- Placez les tags juste avant le texte qu'ils doivent affecter
- Evitez de surcharger avec trop de tags dans une meme phrase
- `[pause]` est utile pour creer du suspense
- Fish S2 Pro via API cloud ne necessite pas de GPU

## Criteres de succes

- [ ] Texte narratif avec 3+ tags expressifs differents
- [ ] 2 versions generees (defaut + clonee ou 2 styles differents)
- [ ] Evaluation comparative avec scores justifies
- [ ] Fichiers audio sauvegardes

---

**Soumission** : PR avec titre "Exercice TTS Expressif - [Votre Nom]", texte, fichiers audio et evaluation

---

## Conclusion

Ce notebook a exploré l'évolution du TTS vers l'expressivité et le contrôle granulaire de la prosodie. Les modèles SOTA de 2024-2025 comme Fish S2 Pro et Dia TTS marquent un tournant dans la synthèse vocale.

### Récapitulatif des apprentissages

| Concept | Points clés |
|---------|-------------|
| **Tags d'expressivité** | 15000+ tags (Fish) : `[laugh]`, `[whisper]`, `[pause]`, émotions, tons professionnels |
| **Architecture Dual-AR** | Slow AR (4B) + Fast AR (400M) pour génération rapide et haute qualité |
| **Voice cloning** | Zero-shot avec 10-30s de référence, cross-lingual |
| **Multi-speaker** | Support natif (Dia) ou via voice cloning (Fish) |
| **Multilingue** | 80+ langues avec détection automatique, hiérarchie de qualité |

### Choix du modèle selon le cas d'usage

| Cas d'usage | Modèle recommandé | Raison |
|-------------|-------------------|--------|
| Production haut de gamme | Fish S2 Pro | Qualité SOTA, tags exhaustifs |
| Dialogues temps réel | Dia TTS | Multi-speaker natif, VRAM réduite |
| Applications embarquées | Kokoro | 82M paramètres, latence minimale |
| Voice cloning | Fish S2 Pro | Meilleure fidélité, cross-lingual |
| Usage commercial | Dia TTS / Kokoro | Licence Apache 2.0 |

### Limites actuelles

1. **VRAM** : Fish S2 Pro nécessite 14-18 GB VRAM (solution : API cloud)
2. **Latence** : Temps de génération de plusieurs secondes pour les longs textes
3. **Langues Tier 3** : Qualité variable pour les langues peu représentées
4. **Coût** : API cloud payante au-delà du quota gratuit

### Perspectives d'avenir

- **Modèles plus légers** : Distillation pour usage edge/mobile
- **Temps réel** : Streaming audio avec < 100ms latence
- **Contrôle accru** : Paramètres explicites (émotion, intensité, tempo)
- **Audio3D** : Synthèse spatiale pour VR/AR

### Continuité pédagogique

Les compétences acquises dans ce notebook sont directement réutilisables dans :
- **03-2** : Pipeline vocal complet (ASR + TTS + traitement)
- **04-1** : Narration éducative automatisée
- **04-2** : Accessibilité (lecteurs d'écran expressifs)
- Projets de doublage, podcasting, voicebots

